<a href="https://colab.research.google.com/github/tomzavi56/South-Africa-Labour-Market-Analysis/blob/main/02_data_cleaning_and_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
#importing the relevant packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#loading the data from statsSa
import os
from google.colab import files

filename = "QLFS Trends 2008-2026Q2.xlsx"

if os.path.exists(filename):
    print("File already uploaded:", filename)
else:
    uploaded = files.upload()
    filename = next(iter(uploaded))
    print("Uploaded file:", filename)

raw = pd.read_excel(
    filename,
    sheet_name="Table 2",
    header=None
)

print("Dataset loaded successfully.")
print("Shape:", raw.shape)

# Extracting quarterly labels
quarters = raw.iloc[1, 1:]

# Extracting official unemployment rate
rates = pd.to_numeric(raw.iloc[24, 1:], errors="coerce")

# Creating dataframe
unemployment_df = pd.DataFrame({
    "quarter": quarters.to_numpy(),
    "unemployment_rate": rates.to_numpy()
})

display(unemployment_df.head())

# Checking the size of the extracted dataset
print("Shape of unemployment dataset:", unemployment_df.shape)

# Checking for missing values
print("\nMissing values:")
print(unemployment_df.isnull().sum())

# Checking for duplicate rows
print("\nDuplicated rows:")
print(unemployment_df.duplicated().sum())

# Checking for duplicated quarters
print("\nDuplicated quarters:")
print(unemployment_df["quarter"].duplicated().sum())

# Checking data types
print("\nData types:")
print(unemployment_df.dtypes)

# Creating year and quarter variables

quarter_mapping = {
    "Jan-Mar": "Q1",
    "Apr-Jun": "Q2",
    "Jul-Sep": "Q3",
    "Oct-Dec": "Q4"
}

# Separating the quarter name and year
parts = unemployment_df["quarter"].str.extract(
    r"(Jan-Mar|Apr-Jun|Jul-Sep|Oct-Dec)\s+(\d{4})"
)

unemployment_df["year"] = parts[1].astype(int)

unemployment_df["quarter_number"] = (
    parts[0].map(quarter_mapping)
)

# Creating a cleaner period variable
unemployment_df["period"] = (
    unemployment_df["year"].astype(str)
    + unemployment_df["quarter_number"]
)

display(unemployment_df.head())

# Validatimg the reported unemployment rate

labour_force = pd.to_numeric(
    raw.iloc[6, 1:],
    errors="coerce"
)

unemployed = pd.to_numeric(
    raw.iloc[11, 1:],
    errors="coerce"
)

calculated_rate = (
    unemployed / labour_force
) * 100

validation_df = pd.DataFrame({
    "period": unemployment_df["period"],
    "reported_rate": unemployment_df["unemployment_rate"],
    "calculated_rate": calculated_rate.to_numpy()
})

validation_df["difference"] = (
    validation_df["reported_rate"]
    - validation_df["calculated_rate"]
).abs()

display(validation_df.head())

print(
    "\nMaximum absolute difference:",
    validation_df["difference"].max()
)

# Creating final cleaned dataset

unemployment_clean = unemployment_df[
    [
        "period",
        "year",
        "quarter_number",
        "unemployment_rate"
    ]
].copy()

unemployment_clean = unemployment_clean.reset_index(drop=True)

print("Clean dataset shape:", unemployment_clean.shape)

display(unemployment_clean.head())
display(unemployment_clean.tail())


# Creating a proper quarterly date variable

unemployment_clean["date"] = pd.PeriodIndex(
    unemployment_clean["period"],
    freq="Q"
).to_timestamp()

# Reordering the columns
unemployment_clean = unemployment_clean[
    [
        "date",
        "period",
        "year",
        "quarter_number",
        "unemployment_rate"
    ]
]

display(unemployment_clean.head())
display(unemployment_clean.tail())

# Saving the cleaned unemployment dataset

unemployment_clean.to_csv(
    "south_africa_unemployment_2008_2026.csv",
    index=False
)

print("Cleaned dataset saved successfully.")

# Checking the saved file

import os

print(
    os.path.exists(
        "south_africa_unemployment_2008_2026.csv"
    )
)

# Downloading the cleaned dataset

from google.colab import files

files.download(
    "south_africa_unemployment_2008_2026.csv"
)

File already uploaded: QLFS Trends 2008-2026Q2.xlsx
Dataset loaded successfully.
Shape: (80, 75)


,quarter,unemployment_rate
0,Jan-Mar 2008,23.2
1,Apr-Jun 2008,22.6
2,Jul-Sep 2008,22.8
3,Oct-Dec 2008,21.5
4,Jan-Mar 2009,23.0


Shape of unemployment dataset: (74, 2)

Missing values:
quarter              0
unemployment_rate    0
dtype: int64

Duplicated rows:
0

Duplicated quarters:
0

Data types:
quarter               object
unemployment_rate    float64
dtype: object


,quarter,unemployment_rate,year,quarter_number,period
0,Jan-Mar 2008,23.2,2008,Q1,2008Q1
1,Apr-Jun 2008,22.6,2008,Q2,2008Q2
2,Jul-Sep 2008,22.8,2008,Q3,2008Q3
3,Oct-Dec 2008,21.5,2008,Q4,2008Q4
4,Jan-Mar 2009,23.0,2009,Q1,2009Q1


,period,reported_rate,calculated_rate,difference
0,2008Q1,23.2,23.238120,0.038120
1,2008Q2,22.6,22.633616,0.033616
2,2008Q3,22.8,22.810726,0.010726
3,2008Q4,21.5,21.512335,0.012335
4,2009Q1,23.0,23.002606,0.002606


...

,date,period,year,quarter_number,unemployment_rate
0,2008-01-01,2008Q1,2008,Q1,23.2
1,2008-04-01,2008Q2,2008,Q2,22.6
2,2008-07-01,2008Q3,2008,Q3,22.8
3,2008-10-01,2008Q4,2008,Q4,21.5
4,2009-01-01,2009Q1,2009,Q1,23.0


,date,period,year,quarter_number,unemployment_rate
69,2025-04-01,2025Q2,2025,Q2,33.2
70,2025-07-01,2025Q3,2025,Q3,31.9
71,2025-10-01,2025Q4,2025,Q4,31.4
72,2026-01-01,2026Q1,2026,Q1,32.7
73,2026-04-01,2026Q2,2026,Q2,33.6


Cleaned dataset saved successfully.
True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>